# Yol Hasarı Tespiti: Model Eğitimi (Google Colab)

Bu notebook RDD2022 verisiyle bir YOLO11 modelini eğitir, modelin hiç görmediği test bölümünde ölçer,
hata analizini çıkarır ve sunum için gereken tüm grafikleri Google Drive'a kaydeder.

**Başlamadan önce:** `Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU`

| Adım | Süre (T4, tahmini) |
|---|---|
| Veri indirme + dönüştürme | ~10-15 dk |
| Hızlı deneme (`QUICK=True`) | ~5 dk |
| Tam eğitim | saatler sürebilir; ilk epoch süresine bakıp `EPOCHS`'u ayarlayın. Oturum koparsa aynı hücreyi tekrar çalıştırın, Drive'daki son noktadan devam eder. |

Veri: Arya ve ark. (2022), *RDD2022: A multi-national image dataset for automatic Road Damage Detection*, figshare, CC BY 4.0.

## 1. Ortam

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/yol-hasar-tespit'
import os; os.makedirs(DRIVE_DIR, exist_ok=True)

In [ ]:
# Kendi GitHub deponuzun adresini yazın
REPO_URL = 'https://github.com/mahmutsamiseyhan/yol-hasar-tespit.git'

!git clone -q {REPO_URL} /content/yol-hasar-tespit || (cd /content/yol-hasar-tespit && git pull -q)
%cd /content/yol-hasar-tespit
!pip install -q ultralytics remotezip

## 2. Veri: RDD2022 indir ve YOLO formatına çevir

Varsayılan ülkeler: Çekya, Hindistan, Japonya, ABD, Çin (motosiklet). Norveç (10.6 GB, çok yüksek çözünürlük)
ve Çin drone (kuşbakışı, araç kamerası senaryosuna uymuyor) atlanır. Yalnızca seçilen ülkeler indirilir (~2.6 GB).

RDD2022'nin resmi test bölümü **etiketsiz** olduğu için etiketli veri ülke bazında 80/10/10 olarak
train/val/test'e ayrılır. Test metrikleri modelin hiç görmediği görüntülerden ölçülür.

In [ ]:
!python scripts/download_rdd2022.py --out /content/rdd_raw
!python scripts/prepare_rdd2022.py --src /content/rdd_raw --out /content/rdd_yolo
DATA_YAML = '/content/rdd_yolo/data.yaml'

In [ ]:
# Sunum için: sınıf dağılımı grafiği
import json, matplotlib.pyplot as plt
stats = json.load(open('/content/rdd_yolo/stats.json'))
names = {'D00': 'Boyuna çatlak', 'D10': 'Enine çatlak', 'D20': 'Timsah sırtı', 'D40': 'Çukur'}
train = stats['train']['sinif_ornekleri']
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar([f"{k}\n{names[k]}" for k in train], train.values(), color='#0b5394')
for i, v in enumerate(train.values()):
    ax.text(i, v, f'{v:,}', ha='center', va='bottom')
ax.set_title('Eğitim verisinde sınıf dağılımı'); ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.savefig(f'{DRIVE_DIR}/sinif_dagilimi.png', dpi=150); plt.show()
print({s: v['goruntu'] for s, v in stats.items()})

## 3. Eğitim

- `yolo11s`: doğruluk/hız dengesi iyi, araç içi cihazlarda ve CPU sunucuda çalışabilecek boyutta.
- Önce `QUICK=True` ile 5 dakikalık bir deneme yapıp hattın çalıştığını görün, sonra `False` yapıp tam eğitimi başlatın.
- Çıktılar doğrudan Drive'a yazılır. Oturum koparsa bu hücreyi tekrar çalıştırmanız yeterli: son kayıttan devam eder.

In [ ]:
from pathlib import Path
from ultralytics import YOLO

QUICK = True          # İlk denemede True, sonra False
MODEL = 'yolo11s.pt'
EPOCHS = 3 if QUICK else 80
RUN_NAME = 'deneme' if QUICK else 'rdd2022_yolo11s'
PROJECT = f'{DRIVE_DIR}/runs'

last = Path(PROJECT) / RUN_NAME / 'weights' / 'last.pt'
if last.exists() and not QUICK:
    print('Yarım kalan eğitim bulundu, devam ediliyor:', last)
    model = YOLO(str(last))
    model.train(resume=True)
else:
    model = YOLO(MODEL)
    model.train(
        data=DATA_YAML, epochs=EPOCHS, imgsz=640, batch=16, patience=20,
        fraction=0.1 if QUICK else 1.0,
        project=PROJECT, name=RUN_NAME, exist_ok=True,
        cos_lr=True, close_mosaic=10, seed=42, plots=True,
    )
# Ultralytics'in gerçekte kullandığı klasör (sürüme göre yol yapısı değişebiliyor)
RUN_DIR = Path(model.trainer.save_dir)
BEST = RUN_DIR / 'weights' / 'best.pt'
print('En iyi ağırlıklar:', BEST)

In [ ]:
# Eğitim grafikleri (sunuma doğrudan konabilir)
from IPython.display import Image, display
for name in ['results.png', 'confusion_matrix_normalized.png', 'BoxPR_curve.png', 'PR_curve.png']:
    p = RUN_DIR / name
    if p.exists():
        print(name); display(Image(filename=str(p), width=900))

## 4. Test bölümünde değerlendirme

Burada ölçülen değerler sunumda söyleyeceğiniz rakamlardır. Kendi ölçtüğünüz rakamı söyleyin, yuvarlamayın.
Karşılaştırma için: RDD2022 üzerinde YOLOv7 ile 4 sınıfta mAP@0.5 = %63.9 raporlayan açık kaynak bir çalışma var
(github.com/oracl4/RoadDamageDetection). Farklı veri bölümleri kullanıldığı için birebir karşılaştırma değildir.

In [ ]:
import pandas as pd
best_model = YOLO(str(BEST))
m = best_model.val(data=DATA_YAML, split='test', imgsz=640, batch=16, plots=True,
                   project=PROJECT, name=f'{RUN_NAME}_test', exist_ok=True)

rows = []
for i, cls_id in enumerate(m.box.ap_class_index):  # yalnızca test setinde bulunan sınıflar
    c = m.names[int(cls_id)]
    p, r, ap50, ap = m.box.class_result(i)
    rows.append({'sinif': c, 'ad': names.get(c, c), 'precision': p, 'recall': r, 'AP50': ap50, 'AP50-95': ap})
rows.append({'sinif': 'TÜMÜ', 'ad': '', 'precision': m.box.mp, 'recall': m.box.mr, 'AP50': m.box.map50, 'AP50-95': m.box.map})
test_table = pd.DataFrame(rows).round(3)
display(test_table)

metrics = {
    'test': test_table.to_dict(orient='records'),
    'hiz_ms': m.speed,   # görüntü başına ön-işleme / çıkarım / son-işleme (ms, T4)
    'goruntu_sayisi': stats['test']['goruntu'],
}
json.dump(metrics, open(f'{DRIVE_DIR}/test_metrikleri.json', 'w'), indent=2, ensure_ascii=False)
print('Hız (ms/görüntü):', m.speed)

## 5. Hata analizi: model nerede yanılıyor?

Test görüntülerinde tahminler etiketlerle karşılaştırılır (IoU ≥ 0.5, aynı sınıf).
En yüksek güvenli **yanlış pozitifler** (model hasar dedi, etiket yok) ve **kaçırılanlar** gösterilir.
Tipik nedenler: rögar kapakları, yama yapılmış asfalt, gölgeler, ıslak zemin, yol çizgileri.
Bu slayt jüriye "sistemin sınırlarını biliyorum" mesajını verir.

In [ ]:
import numpy as np, cv2, random

def iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1]); x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter + 1e-9)

def load_gt(img_path, w, h):
    lbl = Path(str(img_path).replace('/images/', '/labels/')).with_suffix('.txt')
    out = []
    for line in lbl.read_text().split('\n') if lbl.exists() else []:
        if line.strip():
            c, x, y, bw, bh = map(float, line.split())
            out.append((int(c), [(x-bw/2)*w, (y-bh/2)*h, (x+bw/2)*w, (y+bh/2)*h]))
    return out

test_imgs = sorted(Path('/content/rdd_yolo/images/test').glob('*.jpg'))
fps, fns = [], []
for res in best_model.predict([str(p) for p in test_imgs], conf=0.35, stream=True, verbose=False):
    h, w = res.orig_shape
    gt = load_gt(res.path, w, h); matched = set()
    for box, c, conf in zip(res.boxes.xyxy.tolist(), res.boxes.cls.tolist(), res.boxes.conf.tolist()):
        hit = next((j for j, (gc, gb) in enumerate(gt) if j not in matched and gc == int(c) and iou(box, gb) >= 0.5), None)
        if hit is None: fps.append((conf, res.path, box, m.names[int(c)]))
        else: matched.add(hit)
    for j, (gc, gb) in enumerate(gt):
        if j not in matched: fns.append((res.path, gb, m.names[gc]))

print(f'Yanlış pozitif: {len(fps)}, kaçırılan: {len(fns)}')

def show(items, title, color, n=8):
    fig, axes = plt.subplots(2, 4, figsize=(16, 8)); fig.suptitle(title)
    for ax, it in zip(axes.flat, items[:n]):
        path, box, label = (it[1], it[2], it[3]) if len(it) == 4 else it
        img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        x1, y1, x2, y2 = map(int, box); cv2.rectangle(img, (x1, y1), (x2, y2), color, 4)
        ax.imshow(img); ax.set_title(f'{label}' + (f' %{it[0]*100:.0f}' if len(it) == 4 else '')); ax.axis('off')
    plt.tight_layout(); plt.savefig(f'{DRIVE_DIR}/{title.split(":")[0].replace(" ", "_").lower()}.png', dpi=110); plt.show()

show(sorted(fps, reverse=True), 'Yanlis pozitifler: model hasar dedi, etiket yok', (255, 0, 0))
random.seed(0); show(random.sample(fns, min(8, len(fns))), 'Kacirilanlar: etiket var, model bulamadi', (255, 200, 0))

## 6. Bursa test seti (alan farkı / domain gap ölçümü)

RDD2022'de Türkiye verisi yok. Modelin Bursa yollarındaki gerçek performansı ayrıca ölçülmelidir.

1. `scripts/fetch_mapillary_bursa.py` ile Mapillary'den Bursa görüntüleri indirin ve/veya kendiniz fotoğraf çekin.
2. Aşağıdaki **model destekli ön-etiketleme** hücresiyle modelin tahminlerini YOLO etiketi olarak üretin.
3. Görüntü + etiketleri CVAT veya Roboflow'a yükleyip düzeltin (sıfırdan çizmekten çok daha hızlı).
4. Düzeltilmiş seti `DRIVE_DIR/bursa_test/` altına `images/` + `labels/` olarak koyun ve değerlendirme hücresini çalıştırın.

In [ ]:
# Ön-etiketleme: Bursa görüntülerini Drive'da DRIVE_DIR/bursa_raw/ klasörüne koyun
BURSA_RAW = Path(DRIVE_DIR) / 'bursa_raw'
if BURSA_RAW.exists():
    best_model.predict(str(BURSA_RAW), conf=0.25, save_txt=True, save=True,
                       project=DRIVE_DIR, name='bursa_on_etiket', exist_ok=True, verbose=False)
    print('Ön-etiketler:', f'{DRIVE_DIR}/bursa_on_etiket/labels  (CVAT/Roboflow\'a YOLO formatında yükleyin)')
else:
    print('Bursa ham görüntüleri bulunamadı:', BURSA_RAW)

In [ ]:
# Değerlendirme: RDD test seti vs Bursa test seti
BURSA = Path(DRIVE_DIR) / 'bursa_test'
if (BURSA / 'images').exists():
    (BURSA / 'data.yaml').write_text(
        f"path: {BURSA}\ntrain: images\nval: images\ntest: images\nnames:\n  0: D00\n  1: D10\n  2: D20\n  3: D40\n")
    mb = best_model.val(data=str(BURSA / 'data.yaml'), split='test', imgsz=640, plots=True,
                        project=PROJECT, name=f'{RUN_NAME}_bursa', exist_ok=True)
    compare = pd.DataFrame({
        'RDD2022 test': [m.box.mp, m.box.mr, m.box.map50, m.box.map],
        'Bursa test': [mb.box.mp, mb.box.mr, mb.box.map50, mb.box.map],
    }, index=['precision', 'recall', 'mAP50', 'mAP50-95']).round(3)
    display(compare)
    metrics['bursa'] = compare.to_dict()
    json.dump(metrics, open(f'{DRIVE_DIR}/test_metrikleri.json', 'w'), indent=2, ensure_ascii=False)
else:
    print('Bursa test seti henüz yok:', BURSA)

## 7. Dışa aktarma

`best.pt` dosyasını Drive'dan indirip projede `models/best.pt` konumuna koyun.
ONNX çıktısı, PyTorch kurulu olmayan sunucularda veya araç içi cihazlarda çalıştırmak içindir.

In [ ]:
import shutil
shutil.copy(BEST, f'{DRIVE_DIR}/best.pt')
onnx_path = best_model.export(format='onnx', imgsz=640, dynamic=False, simplify=True)
shutil.copy(onnx_path, f'{DRIVE_DIR}/best.onnx')
print('Drive\'a kaydedildi:', f'{DRIVE_DIR}/best.pt', f'{DRIVE_DIR}/best.onnx')

In [ ]:
# Örnek tahminler (sunum için)
sample = random.Random(1).sample(test_imgs, 8)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, res in zip(axes.flat, best_model.predict([str(p) for p in sample], conf=0.35, verbose=False)):
    ax.imshow(res.plot()[:, :, ::-1]); ax.axis('off')
plt.tight_layout(); plt.savefig(f'{DRIVE_DIR}/ornek_tahminler.png', dpi=110); plt.show()